In [ ]:
# Typically we don't need to reinstall dependencies but because this is collab we have to
!pip -q install pandas tqdm trafilatura
!pip install feedparser

In [ ]:
import re
import time
import pandas as pd
import feedparser
from tqdm.auto import tqdm

# -----------------------------
# BROADER MARKETING SUBREDDITS
# -----------------------------

# I tried to include more broader subreddits, but they still need to be generally niche for the most data collection.
# aka they have to be somewhat popular amoung users

# trying to create a fn that finds subreddits with keywords in the titles; instead of hardcoded names of subreddits

SUBREDDITS = [
    # Convenience / meal planning
    "EatCheapAndHealthy",
    "MealPrepSunday",
    "15minutefood",
    "easyrecipes",
    "Cooking",
    "cookingforbeginners",
    "onepotmeals",
    "slowcooking",
    "InstantPot",

    # Budget + students
    "Frugal",
    "povertyfinance",
    "college",

    # Health goals
    "loseit",
    "Volumeeating",
    "HealthyFood",

    # Diet restrictions
    "glutenfree",
    "Celiac",
    "dairyfree",
    "foodallergies",

    # Travel/minimal kitchen
    "travel",
    "Solotravel",
    "digitalnomad",
    "vanlife",
    "camping",

    # Curry fit / flavor communities
    "IndianFood",
    "spicy",
    "recipes",
]

# -----------------------------
# BROADER PAIN DETECTOR
# -----------------------------

# Additionally, I also tried to make the pains more varying so we could try to reach a bigger audience

PAIN_RE = re.compile(
    r"(help|advice|suggest|ideas|what do i|where can i|recommend|"
    r"quick|easy|fast|weeknight|lazy|simple|"
    r"meal prep|batch|make ahead|leftovers|"
    r"budget|cheap|afford|expensive|"
    r"high protein|protein|healthy|calories|macro|"
    r"bored|tired of|same meals|variety|new meals|"
    r"spicy|flavor|taste|bland|"
    r"no kitchen|no microwave|no stove|no fridge|"
    r"allerg|celiac|gluten|dairy|nut|intolerant|"
    r"travel|road trip|camp|vanlife|hostel|hotel|family|kids)",
    re.I
)

CACHE_PATH = "kays_broad_reddit_cache.csv"

In [ ]:
def load_cache(path=CACHE_PATH):
    try:
        df = pd.read_csv(path)
        if "url" in df.columns:
            return df, set(df["url"].astype(str))
    except Exception:
        pass
    return pd.DataFrame(), set()


def fetch_rss(subreddit: str):
    feed_url = f"https://www.reddit.com/r/{subreddit}/.rss"
    d = feedparser.parse(feed_url)
    rows = []

    for e in d.entries:
        title = (e.get("title") or "").strip()
        summary = (e.get("summary") or "").strip()
        url = (e.get("link") or "").strip()

        text = f"{title} {summary}".strip()

        if not url:
            continue

        if PAIN_RE.search(text):
            rows.append({
                "subreddit": subreddit,
                "url": url,
                "title": title,
                "text_blob": text
            })

    return rows

In [ ]:
existing_df, seen_urls = load_cache()

rows = []

for sr in tqdm(SUBREDDITS, desc="Fetching RSS"):
    new_rows = fetch_rss(sr)

    for r in new_rows:
        if r["url"] not in seen_urls:
            rows.append(r)

    time.sleep(0.25)

new_df = pd.DataFrame(rows).drop_duplicates(subset=["url"]).reset_index(drop=True)

if not existing_df.empty:
    df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(subset=["url"])
else:
    df = new_df

df.to_csv(CACHE_PATH, index=False)

print("Total cached rows:", len(df))
df.head()

Fetching RSS:   0%|          | 0/27 [00:00<?, ?it/s]

Total cached rows: 568


,subreddit,url,title,text_blob
0,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,MOD PSA - This forum is NOT for seeking medica...,MOD PSA - This forum is NOT for seeking medica...
1,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,"[MOD POST] Before you post, asking questions f...","[MOD POST] Before you post, asking questions f..."
2,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,White rice additives,White rice additives <!-- SC_OFF --><div class...
3,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,Recipes with “sneaked in” veggies?,Recipes with “sneaked in” veggies? <!-- SC_OFF...
4,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,Replacements for Chili Oil,Replacements for Chili Oil <!-- SC_OFF --><div...


In [ ]:
def segment_from_title(title: str) -> str:
    t = (title or "").lower()

    if any(k in t for k in ["meal prep", "batch", "make ahead"]):
        return "Meal Prep"
    if any(k in t for k in ["budget", "cheap", "afford", "frugal"]):
        return "Budget"
    if any(k in t for k in ["college", "student", "dorm"]):
        return "Students"
    if any(k in t for k in ["no kitchen", "no microwave", "no stove", "no fridge"]):
        return "No Kitchen"
    if any(k in t for k in ["gluten", "celiac", "allerg", "dairyfree"]):
        return "Diet Restriction"
    if any(k in t for k in ["travel", "road trip", "hostel", "hotel", "vanlife", "camp"]):
        return "Travel"
    if any(k in t for k in ["protein", "macros", "calories", "lose weight", "healthy"]):
        return "Health/Fitness"
    if any(k in t for k in ["weeknight", "quick", "easy", "fast"]):
        return "Busy Weeknights"
    if any(k in t for k in ["spicy", "flavor", "bland"]):
        return "Flavor Seekers"

    return "General"


df["segment"] = df["title"].apply(segment_from_title)
df.head()

,subreddit,url,title,text_blob,segment
0,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,MOD PSA - This forum is NOT for seeking medica...,MOD PSA - This forum is NOT for seeking medica...,General
1,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,"[MOD POST] Before you post, asking questions f...","[MOD POST] Before you post, asking questions f...",General
2,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,White rice additives,White rice additives <!-- SC_OFF --><div class...,General
3,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,Recipes with “sneaked in” veggies?,Recipes with “sneaked in” veggies? <!-- SC_OFF...,General
4,EatCheapAndHealthy,https://www.reddit.com/r/EatCheapAndHealthy/co...,Replacements for Chili Oil,Replacements for Chili Oil <!-- SC_OFF --><div...,General


In [ ]:
def kays_priority(title: str) -> int:
    t = (title or "").lower()
    strong = [
        "no kitchen", "no microwave", "no stove",
        "meal prep", "batch",
        "budget", "cheap",
        "gluten", "celiac",
        "weeknight", "quick", "easy"
    ]
    return int(any(s in t for s in strong))


df["kays_priority"] = df["title"].apply(kays_priority)

df = df.sort_values(by=["kays_priority"], ascending=False).reset_index(drop=True)

df.head(20)

,subreddit,url,title,text_blob,segment,kays_priority
0,recipes,https://www.reddit.com/r/recipes/comments/1r0j...,Fettuccine Alfredo Easy Italian Cheese & Butte...,Fettuccine Alfredo Easy Italian Cheese & Butte...,Busy Weeknights,1
1,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Simple meal prep,"Simple meal prep <table> <tr><td> <a href=""htt...",Meal Prep,1
2,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal Prep Monday: Protein Pasta & Snackboxes,Meal Prep Monday: Protein Pasta & Snackboxes <...,Meal Prep,1
3,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prep for my fiancée and I for the week!,Meal prep for my fiancée and I for the week! <...,Meal Prep,1
4,Frugal,https://www.reddit.com/r/Frugal/comments/1r6kr...,Clubbing/partying is actually such a cheap hob...,Clubbing/partying is actually such a cheap hob...,Budget,1
5,Celiac,https://www.reddit.com/r/Celiac/comments/1r7go...,navigating a possible celiac diagnosis,navigating a possible celiac diagnosis <!-- SC...,Diet Restriction,1
6,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prep as a uni student,Meal prep as a uni student <table> <tr><td> <a...,Meal Prep,1
7,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prep ideas using Indian groceries?,Meal prep ideas using Indian groceries? <!-- S...,Meal Prep,1
8,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prepped some bibimbap in Souper Cubes to ...,Meal prepped some bibimbap in Souper Cubes to ...,Meal Prep,1
9,HealthyFood,https://www.reddit.com/r/HealthyFood/comments/...,Easy high protein breakfast,Easy high protein breakfast <table> <tr><td> <...,Health/Fitness,1


In [ ]:
df_kays = df[df["kays_priority"] == 1].reset_index(drop=True)

df_kays.to_csv("kays_priority_leads.csv", index=False)

df_kays.head(30)

,subreddit,url,title,text_blob,segment,kays_priority
0,recipes,https://www.reddit.com/r/recipes/comments/1r0j...,Fettuccine Alfredo Easy Italian Cheese & Butte...,Fettuccine Alfredo Easy Italian Cheese & Butte...,Busy Weeknights,1
1,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Simple meal prep,"Simple meal prep <table> <tr><td> <a href=""htt...",Meal Prep,1
2,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal Prep Monday: Protein Pasta & Snackboxes,Meal Prep Monday: Protein Pasta & Snackboxes <...,Meal Prep,1
3,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prep for my fiancée and I for the week!,Meal prep for my fiancée and I for the week! <...,Meal Prep,1
4,Frugal,https://www.reddit.com/r/Frugal/comments/1r6kr...,Clubbing/partying is actually such a cheap hob...,Clubbing/partying is actually such a cheap hob...,Budget,1
5,Celiac,https://www.reddit.com/r/Celiac/comments/1r7go...,navigating a possible celiac diagnosis,navigating a possible celiac diagnosis <!-- SC...,Diet Restriction,1
6,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prep as a uni student,Meal prep as a uni student <table> <tr><td> <a...,Meal Prep,1
7,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prep ideas using Indian groceries?,Meal prep ideas using Indian groceries? <!-- S...,Meal Prep,1
8,MealPrepSunday,https://www.reddit.com/r/MealPrepSunday/commen...,Meal prepped some bibimbap in Souper Cubes to ...,Meal prepped some bibimbap in Souper Cubes to ...,Meal Prep,1
9,HealthyFood,https://www.reddit.com/r/HealthyFood/comments/...,Easy high protein breakfast,Easy high protein breakfast <table> <tr><td> <...,Health/Fitness,1


In [ ]:
# Segment distribution
segment_counts = df["segment"].value_counts()

# Top subreddits
subreddit_counts = df["subreddit"].value_counts()

print("Top Segments:")
print(segment_counts.head(10))

print("\nTop Subreddits:")
print(subreddit_counts.head(10))

Top Segments:
segment
General             422
Diet Restriction     32
Travel               29
Health/Fitness       22
Busy Weeknights      18
Budget               16
Meal Prep            12
Students             11
Flavor Seekers        6
Name: count, dtype: int64

Top Subreddits:
subreddit
glutenfree            29
travel                29
Celiac                27
vanlife               26
camping               26
HealthyFood           25
foodallergies         25
EatCheapAndHealthy    25
dairyfree             25
spicy                 25
Name: count, dtype: int64
